<a href="https://colab.research.google.com/github/ShreyaMathur19/Clustered-Diagonal-Segment-Format-CDSF-/blob/main/DIA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**DIA (Diagonal Storage)**

DIA stores a fixed number of diagonals in a dense 2D array with corresponding offsets, including padding where necessary. We performed SpMV and measured execution time and memory usage to analyze its performance for matrices with few diagonals

In [ ]:
from scipy.io import mmread
import numpy as np
from scipy.sparse import coo_matrix
from time import perf_counter
import numba as nb

In [ ]:
# Load the matrix from an .mtx file
A = mmread("wang3.mtx")
A = coo_matrix(A)

In [ ]:
# Convert COO to DIA
A_dia = A.todia()

In [ ]:
# Space Requirement
data_bytes = A_dia.data.nbytes        # storage for diagonal values
offset_bytes = A_dia.offsets.nbytes   # storage for offsets
total_bytes = data_bytes + offset_bytes

In [ ]:
# Conversion time during conversion from COO to DIA
def time_and_peak_mem_coo_to_dia(A_coo, reps=5, interval=0.001):
    """
    Measures average time (ms) and peak memory (MB) for A_coo.todia().
    Uses memory_profiler to measure peak RSS per run, then reports the max peak.
    """
    times_ms = []
    peak_deltas_mb = []
    A_dia = None

    # One warm-up (not timed) to avoid cold-start noise
    _ = A_coo.todia()

    for _ in range(reps):
        t0 = perf_counter()
        mem_trace, A_dia = memory_usage(
            (_coo_to_dia, (A_coo,), {}), retval=True, interval=interval
        )
        elapsed_ms = (perf_counter() - t0) * 1000.0
        # memory_usage returns a list of RSS (MB); take peak - start as delta for the run
        peak_delta_mb = max(mem_trace) - mem_trace[0]

        times_ms.append(elapsed_ms)
        peak_deltas_mb.append(peak_delta_mb)

    avg_ms = float(np.mean(times_ms))
    peak_mb = float(max(peak_deltas_mb))  # the worst (highest) peak across runs

    print(f"COO->DIA: {avg_ms:.2f} ms per call (avg over {reps})")
    print(f"Peak memory during conversion: {peak_mb:.2f} MB")
    return A_dia, avg_ms, peak_mb



In [ ]:
#SPMV
def time_spmv_dia(A_dia, reps=20, seed=42):
    n = A_dia.shape[1]
    rng = np.random.default_rng(seed)
    x = rng.standard_normal(n).astype(np.float64)

    # warm-up
    _ = A_dia @ x

    t0 = perf_counter()
    for _ in range(reps):
        y = A_dia @ x
    t_ms = (perf_counter() - t0) / reps * 1000.0  # ms per call

    print(f"Shape: {A_dia.shape}, diagonals: {len(A_dia.offsets)}")
    print(f"y = A_dia @ x  : {t_ms:.2f} ms per call")

    return t_ms
